---
# **Cuaderno 4:**  Analisis de datos de Clinvar
---
* Author: Melissa Ormeño; Alexis Salas
* Laboratory: Cancer Molecular Dynamics
* Date: Agosto 2025

**Aims:**

* disease names and gene-disease relationships (https://ftp.ncbi.nlm.nih.gov/pub/clinvar/	)

In [1]:
import pandas as pd

In [2]:
vcf_path = "../data/ClinVar/clinvar.vcf.gz"

cols = ["#CHROM","POS","ID","REF","ALT","QUAL","FILTER","INFO"]  # VCF sin muestras
df = pd.read_csv(
    vcf_path,
    sep="\t",
    comment="#",
    names=cols,
    dtype={"#CHROM":"string","POS":"int64","ID":"string",
           "REF":"string","ALT":"string","QUAL":"string","FILTER":"string","INFO":"string"}
)
df.head()


,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
0,1,66926,3385321,AG,A,.,.,ALLELEID=3544463;CLNDISDB=Human_Phenotype_Onto...
1,1,69134,2205837,A,G,.,.,ALLELEID=2193183;CLNDISDB=MedGen:CN169374;CLND...
2,1,69308,3925305,A,G,.,.,ALLELEID=4039319;CLNDISDB=MedGen:CN169374;CLND...
3,1,69314,3205580,T,G,.,.,ALLELEID=3374047;CLNDISDB=MedGen:CN169374;CLND...
4,1,69404,3925306,T,C,.,.,ALLELEID=4039320;CLNDISDB=MedGen:CN169374;CLND...


In [3]:
# df tiene una columna "INFO" de tipo string
def info_to_wide(info_series):
    dicts = info_series.fillna("").str.split(";").apply(
        lambda parts: {k: v for k, v in
                       (p.split("=", 1) for p in parts if "=" in p)}
    )
    return pd.json_normalize(dicts)  # columnas = claves, filas = variantes

info_df = info_to_wide(df["INFO"])
# Si quieres SOLO las columnas de INFO:
df = info_df


In [4]:
df

,ALLELEID,CLNDISDB,CLNDN,CLNHGVS,CLNREVSTAT,CLNSIG,CLNSIGSCV,CLNVC,CLNVCSO,GENEINFO,...,ONCDISDB,ONCDN,ONCREVSTAT,ONCSCV,SCI,SCIREVSTAT,DBVARID,SCIDISDB,SCIDN,SCISCV
0,3544463,"Human_Phenotype_Ontology:HP:0000547,MONDO:MOND...",Retinitis_pigmentosa,NC_000001.11:g.66927del,"criteria_provided,_single_submitter",Uncertain_significance,SCV005419006,Deletion,SO:0000159,OR4F5:79501,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2193183,MedGen:CN169374,not_specified,NC_000001.11:g.69134A>G,"criteria_provided,_single_submitter",Likely_benign,SCV003526545,single_nucleotide_variant,SO:0001483,OR4F5:79501,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4039319,MedGen:CN169374,not_specified,NC_000001.11:g.69308A>G,"criteria_provided,_single_submitter",Uncertain_significance,SCV006120601,single_nucleotide_variant,SO:0001483,OR4F5:79501,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3374047,MedGen:CN169374,not_specified,NC_000001.11:g.69314T>G,"criteria_provided,_single_submitter",Uncertain_significance,SCV004995495,single_nucleotide_variant,SO:0001483,OR4F5:79501,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4039320,MedGen:CN169374,not_specified,NC_000001.11:g.69404T>C,"criteria_provided,_single_submitter",Uncertain_significance,SCV006120602,single_nucleotide_variant,SO:0001483,OR4F5:79501,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3677945,2232003,MedGen:CN169374,not_specified,NaN,"criteria_provided,_single_submitter",Uncertain_significance,SCV003560205,single_nucleotide_variant,SO:0001483,LILRA3:11026,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3677946,3894028,MedGen:C3661900,not_provided,NaN,"criteria_provided,_single_submitter",Likely_benign,SCV005911038,single_nucleotide_variant,SO:0001483,LILRA3:11026,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3677947,2200058,MedGen:CN169374,not_specified,NaN,"criteria_provided,_single_submitter",Uncertain_significance,SCV003533488,single_nucleotide_variant,SO:0001483,LILRA3:11026,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3677948,2226217,MedGen:CN169374,not_specified,NaN,"criteria_provided,_single_submitter",Uncertain_significance,SCV003570821,single_nucleotide_variant,SO:0001483,LILRA3:11026|LOC126057115:126057115,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df.columns

Index(['ALLELEID', 'CLNDISDB', 'CLNDN', 'CLNHGVS', 'CLNREVSTAT', 'CLNSIG',
       'CLNSIGSCV', 'CLNVC', 'CLNVCSO', 'GENEINFO', 'MC', 'ORIGIN', 'CLNVI',
       'RS', 'AF_EXAC', 'AF_TGP', 'AF_ESP', 'CLNSIGCONF', 'CLNDISDBINCL',
       'CLNDNINCL', 'CLNSIGINCL', 'ONC', 'ONCDISDB', 'ONCDN', 'ONCREVSTAT',
       'ONCSCV', 'SCI', 'SCIREVSTAT', 'DBVARID', 'SCIDISDB', 'SCIDN',
       'SCISCV'],
      dtype='object')

In [50]:
df['SCISCV'].unique()

array([nan, 'SCV004697534', 'SCV004565361', 'SCV005387955',
       'SCV005387958', 'SCV005387959', 'SCV005387957', 'SCV005045674',
       'SCV006083359', 'SCV005045673', 'SCV005045677', 'SCV005045676',
       'SCV005045675', 'SCV004697533', 'SCV005387954', 'SCV005687494',
       'SCV005687766', 'SCV005200115', 'SCV005200116', 'SCV005061371',
       'SCV005061369', 'SCV005061368', 'SCV005061370', 'SCV005061367',
       'SCV005061366', 'SCV005061365', 'SCV005200118', 'SCV005871645',
       'SCV004697527|SCV004697528', 'SCV004697531|SCV004697532'],
      dtype=object)

In [26]:
df_drop = df.dropna(subset=['CLNSIG', 'CLNDN'])

In [27]:
df_loc = df_drop.loc[df_drop['CLNSIG'].str.contains("Pathogenic") | df_drop['CLNSIG'].str.contains("pathogenic")] 
df_loc


,ALLELEID,CLNDISDB,CLNDN,CLNHGVS,CLNREVSTAT,CLNSIG,CLNSIGSCV,CLNVC,CLNVCSO,GENEINFO,...,ONCDISDB,ONCDN,ONCREVSTAT,ONCSCV,SCI,SCIREVSTAT,DBVARID,SCIDISDB,SCIDN,SCISCV
47,1023510,MedGen:C3661900|MedGen:CN169374,not_provided|not_specified,NC_000001.11:g.930200G>A,"criteria_provided,_conflicting_classifications",Conflicting_classifications_of_pathogenicity,SCV001541340|SCV003595924,single_nucleotide_variant,SO:0001483,SAMD11:148398,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
91,987769,MedGen:CN169374|MedGen:C3661900,not_specified|not_provided,NC_000001.11:g.930317A>G,"criteria_provided,_conflicting_classifications",Conflicting_classifications_of_pathogenicity,SCV001487486|SCV004945557,single_nucleotide_variant,SO:0001483,SAMD11:148398,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
173,1067612,MedGen:CN169374|MedGen:C3661900,not_specified|not_provided,NC_000001.11:g.935839C>T,"criteria_provided,_conflicting_classifications",Conflicting_classifications_of_pathogenicity,SCV001605176|SCV003554949,single_nucleotide_variant,SO:0001483,SAMD11:148398,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
240,1441283,MedGen:CN169374|MedGen:C3661900,not_specified|not_provided,NC_000001.11:g.939117G>A,"criteria_provided,_conflicting_classifications",Conflicting_classifications_of_pathogenicity,SCV002206314|SCV003647516,single_nucleotide_variant,SO:0001483,SAMD11:148398,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
271,1512614,MedGen:C3661900|MedGen:CN169374,not_provided|not_specified,NC_000001.11:g.939293G>C,"criteria_provided,_conflicting_classifications",Conflicting_classifications_of_pathogenicity,SCV002121756|SCV005933543,single_nucleotide_variant,SO:0001483,SAMD11:148398,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3677753,24717,"Human_Phenotype_Ontology:HP:0003546,MedGen:C04...",Exercise_intolerance|Mitochondrial_myopathy_wi...,NC_012920.1:m.15615G>A,"criteria_provided,_single_submitter",Likely_pathogenic,SCV000998381,single_nucleotide_variant,SO:0001483,MT-CYB:4519,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3677853,677625,"MONDO:MONDO:0010789,MedGen:C0162671,OMIM:54000...",MELAS_syndrome,NC_012920.1:m.15915G>A,"criteria_provided,_single_submitter",Pathogenic,SCV000993359,single_nucleotide_variant,SO:0001483,MT-TT:4576,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3677857,48174,"Human_Phenotype_Ontology:HP:0001290,MedGen:C18...",Generalized_hypotonia|Neonatal_onset|Sudden_ca...,NC_012920.1:m.15923A>G,reviewed_by_expert_panel,Likely_pathogenic,SCV005187324,single_nucleotide_variant,SO:0001483,MT-TT:4576,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3677899,24609,"MONDO:MONDO:0044970,MeSH:D028361,MedGen:C07516...",Mitochondrial_disease|Myopathy,NC_012920.1:m.15990C>T,reviewed_by_expert_panel,Likely_pathogenic,SCV003915446,single_nucleotide_variant,SO:0001483,MT-TP:4571,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
df_loc.isnull().sum()

ALLELEID             0
CLNDISDB             0
CLNDN                0
CLNHGVS              3
CLNREVSTAT           0
CLNSIG               0
CLNSIGSCV            1
CLNVC                0
CLNVCSO              0
GENEINFO         13399
MC               15626
ORIGIN           13364
CLNVI            45548
RS               56077
AF_EXAC         358520
AF_TGP          422064
AF_ESP          413625
CLNSIGCONF      311963
CLNDISDBINCL    460513
CLNDNINCL       460513
CLNSIGINCL      460513
ONC             460710
ONCDISDB        460710
ONCDN           460710
ONCREVSTAT      460710
ONCSCV          460710
SCI             461086
SCIREVSTAT      461086
DBVARID         460964
SCIDISDB        461086
SCIDN           461086
SCISCV          461086
dtype: int64

In [28]:
df_loc[['MC', 'ORIGIN', 'CLNVI',
       'RS', 'AF_EXAC', 'AF_TGP', 'AF_ESP', 'CLNSIGCONF', 'CLNDISDBINCL',
       'CLNDNINCL', 'CLNSIGINCL', 'ONC', 'ONCDISDB', 'ONCDN', 'ONCREVSTAT',]]

,MC,ORIGIN,CLNVI,RS,AF_EXAC,AF_TGP,AF_ESP,CLNSIGCONF,CLNDISDBINCL,CLNDNINCL,CLNSIGINCL,ONC,ONCDISDB,ONCDN,ONCREVSTAT
47,SO:0001583|missense_variant,1,ClinGen:CA502426,368250686,0.00038,NaN,0.00039,Uncertain_significance(1)|Likely_benign(1),NaN,NaN,NaN,NaN,NaN,NaN,NaN
91,SO:0001583|missense_variant,1,ClinGen:CA502461,779619427,0.00003,NaN,NaN,Uncertain_significance(1)|Likely_benign(1),NaN,NaN,NaN,NaN,NaN,NaN,NaN
173,SO:0001583|missense_variant,1,ClinGen:CA502582,377367586,0.00017,NaN,0.00008,Uncertain_significance(1)|Likely_benign(1),NaN,NaN,NaN,NaN,NaN,NaN,NaN
240,SO:0001583|missense_variant,1,ClinGen:CA502652,200099017,NaN,NaN,NaN,Uncertain_significance(1)|Likely_benign(1),NaN,NaN,NaN,NaN,NaN,NaN,NaN
271,SO:0001583|missense_variant,1,ClinGen:CA337800211,112419559,NaN,NaN,NaN,Uncertain_significance(1)|Likely_benign(1),NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3677753,NaN,1,ClinGen:CA120616|OMIM:516020.0005,207459997,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3677853,NaN,1,ClinGen:CA913175191,1603225588,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3677857,NaN,1,ClinGen:CA130372|OMIM:590090.0002,1556424691,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3677899,NaN,1,ClinGen:CA120549|OMIM:590075.0001,199474699,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
